In [2]:
import os, numpy as np, pandas as pd, torch, gc
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from monai.networks.nets import SwinUNETR, BasicUNet
import segmentation_models_pytorch as smp

# --- PATHS ---
NPY_3D_DIR = r"C:\Users\EGlaciers\Desktop\Project NCKH\BrainMRI\Convert3D"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- HYPERPARAMETERS ---
BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-4
# Lát 77 tương ứng index 27 trong mảng 64 lát (50-114)
MID_IDX = 27

In [3]:
class BraTS2DFrom3DDataset(Dataset):
    def __init__(self, p_ids):
        self.p_ids = p_ids

    def __len__(self): return len(self.p_ids)

    def __getitem__(self, idx):
        p_id = self.p_ids[idx]
        # Load khối 3D: Image (4, 160, 160, 64), Mask (160, 160, 64)
        x_3d = np.load(os.path.join(NPY_3D_DIR, f"{p_id}_img.npy"))
        y_3d = np.load(os.path.join(NPY_3D_DIR, f"{p_id}_mask.npy"))

        # Chỉ lấy lát cắt ở giữa (index 27)
        x_2d = x_3d[:, :, :, MID_IDX] # Kết quả: (4, 160, 160)
        y_2d = y_3d[:, :, MID_IDX]    # Kết quả: (160, 160)

        return torch.from_numpy(x_2d).float(), torch.from_numpy(y_2d).long()

In [4]:
def get_2d_models():
    models = {}

    # 1. UNet++ (Nested UNet)
    models["UNet++"] = smp.UnetPlusPlus(
        encoder_name="resnet34", in_channels=4, classes=4, encoder_weights=None
    ).to(DEVICE)

    # 2. DeepLabV3+
    models["DeepLabV3+"] = smp.DeepLabV3Plus(
        encoder_name="resnet34", in_channels=4, classes=4, encoder_weights=None
    ).to(DEVICE)

    # 3. Swin-UNet (Dùng SwinUNETR bản 2D)
    models["Swin-UNet"] = SwinUNETR(
        in_channels=4, out_channels=4, spatial_dims=2
    ).to(DEVICE)

    return models

In [5]:
from sklearn.metrics import jaccard_score, accuracy_score

def compute_metrics(pred, target):
    p = pred.argmax(1).flatten().cpu().numpy()
    t = target.flatten().cpu().numpy()

    iou = jaccard_score(t, p, average='macro', labels=[1,2,3], zero_division=0)
    dice = (2 * iou) / (1 + iou) if iou > 0 else 0

    # Nhị phân hóa để tính Sens/Spec (Vùng u vs Nền)
    t_bin, p_bin = (t > 0), (p > 0)
    tp = np.sum(t_bin & p_bin)
    tn = np.sum(~t_bin & ~p_bin)
    fp = np.sum(~t_bin & p_bin)
    fn = np.sum(t_bin & ~p_bin)

    sens = tp / (tp + fn + 1e-7)
    spec = tn / (tn + fp + 1e-7)
    acc = accuracy_score(t, p)

    return {"Dice": dice, "IoU": iou, "Sensitivity": sens, "Specificity": spec, "Accuracy": acc}

In [7]:
from monai.losses import DiceCELoss

def run_train_2d():
    # Chia dữ liệu
    p_ids = [f.replace("_img.npy", "") for f in os.listdir(NPY_3D_DIR) if "_img.npy" in f]
    t_ids, v_ids = train_test_split(p_ids, test_size=0.2, random_state=42)

    train_loader = DataLoader(BraTS2DFrom3DDataset(t_ids), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(BraTS2DFrom3DDataset(v_ids), batch_size=4)

    models = get_2d_models()
    criterion = DiceCELoss(softmax=True, to_onehot_y=True)

    for name, model in models.items():
        print(f"\nĐang luyện {name}...")
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        scaler = torch.amp.GradScaler('cuda')
        history = []
        best_dice = 0

        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0
            pbar = tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False)

            for x, y in pbar:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(x), y.unsqueeze(1))
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
                train_loss += loss.item()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            # Validation
            model.eval()
            m_list = []
            with torch.no_grad():
                for xv, yv in val_loader:
                    with torch.amp.autocast('cuda'):
                        out = model(xv.to(DEVICE))
                    m_list.append(compute_metrics(out, yv))

            avg_m = pd.DataFrame(m_list).mean().to_dict()
            avg_m.update({"epoch": epoch+1, "loss": train_loss/len(train_loader)})
            history.append(avg_m)
            pd.DataFrame(history).to_csv(f"log_2D_npy_{name}.csv", index=False)

            if avg_m['Dice'] > best_dice:
                best_dice = avg_m['Dice']; torch.save(model.state_dict(), f"best_2D_npy_{name}.pth")

            print(f"✅ Ep {epoch+1} | Loss: {avg_m['loss']:.4f} | Dice: {avg_m['Dice']:.4f} | IoU: {avg_m['IoU']:.4f}")

    print("\nHoàn thành huấn luyện 3 mô hình 2D!")

# Gọi hàm chạy
run_train_2d()


Đang luyện UNet++...


KeyboardInterrupt: 